In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
INPUT_PATH = str(project_path(get_value('science.tables.result_175', 'result_175.xlsx')))
SHEET_NAME = 0
OUTPUT_DIR = str(project_path(get_value('science.paths.directory_105', 'directory_105')))
SAMPLE_COL = None
NORMALIZE_SAMPLE_ID_TO_LOWER = True
MISSING_RATE_CUTOFF = 0.3
VARIANCE_EPS = 1e-12
USER_DROP_COLS = []

def read_table(path, sheet_name=0):
    ext = os.path.splitext(path)[1].lower()
    if ext in ['.xlsx', '.xls']:
        return pd.read_excel(path, sheet_name=sheet_name)
    if ext in ['.csv', '.gz']:
        return pd.read_csv(path, encoding='utf-8-sig')
    if ext in ['.tsv', '.txt']:
        return pd.read_csv(path, sep='\t', encoding='utf-8-sig')
    raise ValueError(f'Unsupported file type: {ext}')

def normalize_sample_id(x):
    x = str(x).strip()
    if re.fullmatch('-?\\d+\\.0', x):
        x = x[:-2]
    if NORMALIZE_SAMPLE_ID_TO_LOWER:
        x = x.lower()
    return x

def detect_sample_column(df):
    candidates = ['sample', 'sample_id', 'sample id', 'patient', 'patient_id', 'patient id', 'case', 'case_id', 'case id', 'subject', 'subject_id', 'subject id', 'id']
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]
    return df.columns[0]

def is_obvious_metadata_column(col):
    name = str(col).strip().lower()
    exact_metadata = {'response', get_column('outcomes.response', 'Response').lower(), 'orr', 'group', 'label', 'pfs', 'pfs_time', 'pfs_status', 'pfs_event', 'os', 'os_time', 'os_status', 'os_event', 'cohort', 'center', 'centre', 'site', 'batch', 'sex', 'age'}
    if name in exact_metadata:
        return True
    if name.startswith('diagnostics_'):
        return True
    return False
os.makedirs(OUTPUT_DIR, exist_ok=True)
raw = read_table(INPUT_PATH, sheet_name=SHEET_NAME)
raw.columns = [str(c).strip() for c in raw.columns]
print('Raw matrix shape:', raw.shape)
sample_col = SAMPLE_COL if SAMPLE_COL is not None else detect_sample_column(raw)
if sample_col not in raw.columns:
    raise KeyError(f'Sample column not found: {sample_col}')
sample_original = raw[sample_col].copy()
sample_normalized = raw[sample_col].map(normalize_sample_id)
if sample_normalized.isna().any() or sample_normalized.eq('').any():
    raise ValueError('Empty sample IDs were found.')
duplicate_sample_mask = sample_normalized.duplicated(keep=False)
if duplicate_sample_mask.any():
    duplicate_ids = sorted(sample_normalized.loc[duplicate_sample_mask].unique())
    raise ValueError('Duplicate sample IDs were found after normalization:\n' + '\n'.join(map(str, duplicate_ids)))
raw = raw.drop(columns=[sample_col])
raw.index = sample_normalized
raw.index.name = 'sample'
drop_metadata = [c for c in raw.columns if c in USER_DROP_COLS or is_obvious_metadata_column(c)]
X = raw.drop(columns=drop_metadata, errors='ignore').copy()
duplicate_name_mask = pd.Index(X.columns).duplicated(keep='first')
duplicate_name_cols = X.columns[duplicate_name_mask].tolist()
X = X.loc[:, ~duplicate_name_mask]
X_numeric = X.apply(pd.to_numeric, errors='coerce')
all_nan_cols = X_numeric.columns[X_numeric.isna().all()].tolist()
X_numeric = X_numeric.drop(columns=all_nan_cols)
missing_rate = X_numeric.isna().mean(axis=0)
high_missing_cols = missing_rate[missing_rate > MISSING_RATE_CUTOFF].index.tolist()
X_numeric = X_numeric.drop(columns=high_missing_cols)
variance_before = X_numeric.var(axis=0, skipna=True, ddof=0)
zero_var_before_cols = variance_before[variance_before.isna() | (variance_before <= VARIANCE_EPS)].index.tolist()
X_numeric = X_numeric.drop(columns=zero_var_before_cols)
print('After metadata/non-numeric/missingness filtering:', X_numeric.shape)
feature_medians = X_numeric.median(axis=0, skipna=True)
if feature_medians.isna().any():
    bad = feature_medians[feature_medians.isna()].index.tolist()
    raise ValueError(f'Some feature medians are still NaN: {bad[:10]}')
X_imputed = X_numeric.fillna(feature_medians)
variance_after = X_imputed.var(axis=0, ddof=0)
zero_var_after_cols = variance_after[variance_after <= VARIANCE_EPS].index.tolist()
X_imputed = X_imputed.drop(columns=zero_var_after_cols)
feature_medians = feature_medians.loc[X_imputed.columns]
print('After median imputation and zero-variance filtering:', X_imputed.shape)
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled_values = scaler.fit_transform(X_imputed)
X_scaled = pd.DataFrame(X_scaled_values, index=X_imputed.index, columns=X_imputed.columns)
if not np.isfinite(X_scaled.to_numpy()).all():
    raise ValueError('NaN or infinite values remain in the standardized matrix.')
mean_abs_max = float(X_scaled.mean(axis=0).abs().max())
sd_deviation_max = float((X_scaled.std(axis=0, ddof=0) - 1.0).abs().max())
print('Final model-ready matrix:', X_scaled.shape)
print('Maximum absolute feature mean:', mean_abs_max)
print('Maximum absolute SD deviation from 1:', sd_deviation_max)
model_ready = X_scaled.reset_index()
model_ready_path = os.path.join(OUTPUT_DIR, get_value('science.tables.features_002', 'features_002.csv.gz'))
model_ready.to_csv(model_ready_path, index=False, encoding='utf-8-sig', compression='gzip')
parameters = pd.DataFrame({'feature': X_imputed.columns, 'median_internal': feature_medians.loc[X_imputed.columns].values, 'mean_internal_after_imputation': scaler.mean_, 'scale_internal': scaler.scale_, 'variance_internal_after_imputation': scaler.var_})
parameters_path = os.path.join(OUTPUT_DIR, get_value('science.tables.result_176', 'result_176.csv.gz'))
parameters.to_csv(parameters_path, index=False, encoding='utf-8-sig', compression='gzip')
removed_records = []

def add_removed(features, reason):
    for feature in features:
        removed_records.append({'feature': str(feature), 'removal_reason': reason})
add_removed(drop_metadata, 'metadata_or_diagnostics')
add_removed(duplicate_name_cols, 'duplicate_column_name')
add_removed(all_nan_cols, 'non_numeric_or_all_missing')
add_removed(high_missing_cols, f'missing_rate_gt_{MISSING_RATE_CUTOFF}')
add_removed(zero_var_before_cols, 'zero_variance_before_imputation')
add_removed(zero_var_after_cols, 'zero_variance_after_imputation')
removed_df = pd.DataFrame(removed_records, columns=['feature', 'removal_reason'])
removed_path = os.path.join(OUTPUT_DIR, get_value('science.tables.result_177', 'result_177.csv'))
removed_df.to_csv(removed_path, index=False, encoding='utf-8-sig')
sample_qc = pd.DataFrame({'sample': X_numeric.index, 'original_sample_id': sample_original.astype(str).values, 'n_features_before_imputation': X_numeric.shape[1], 'n_missing_before_imputation': X_numeric.isna().sum(axis=1).values, 'missing_fraction_before_imputation': X_numeric.isna().mean(axis=1).values})
sample_qc_path = os.path.join(OUTPUT_DIR, get_value('science.tables.result_178', 'result_178.csv'))
sample_qc.to_csv(sample_qc_path, index=False, encoding='utf-8-sig')
audit = {'input_path': INPUT_PATH, 'sample_column': str(sample_col), 'raw_sample_n': int(len(raw)), 'raw_candidate_column_n': int(raw.shape[1]), 'metadata_or_diagnostics_removed_n': int(len(drop_metadata)), 'duplicate_feature_name_removed_n': int(len(duplicate_name_cols)), 'non_numeric_or_all_missing_removed_n': int(len(all_nan_cols)), 'high_missing_removed_n': int(len(high_missing_cols)), 'zero_variance_before_imputation_removed_n': int(len(zero_var_before_cols)), 'zero_variance_after_imputation_removed_n': int(len(zero_var_after_cols)), 'final_sample_n': int(X_scaled.shape[0]), 'final_feature_n': int(X_scaled.shape[1]), 'missing_rate_cutoff': float(MISSING_RATE_CUTOFF), 'zscore_max_abs_mean': mean_abs_max, 'zscore_max_abs_sd_minus_one': sd_deviation_max, 'high_correlation_filtering': False, 'outcome_labels_used': False}
audit_path = os.path.join(OUTPUT_DIR, get_value('science.tables.result_179', 'result_179.json'))
with open(audit_path, 'w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)








In [ ]:
from pathlib import Path
import json
import re
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', category=RuntimeWarning)
EXTERNAL_INPUT_PATH = Path(str(project_path(get_value('science.tables.result_180', 'result_180.xlsx'))))
SHEET_NAME = 0
PARAMETER_CANDIDATES = [Path(str(project_path(get_value('science.tables.features_047', 'features_047.csv.gz')))), Path(str(project_path(get_value('science.tables.result_181', 'result_181.csv.gz'))))]
INTERNAL_MODEL_READY_CANDIDATES = [Path(str(project_path(get_value('science.tables.features_007', 'features_007.csv.gz')))), Path(str(project_path(get_value('science.tables.features_048', 'features_048.csv.gz'))))]
OUTPUT_MODEL_READY_PATH = Path(str(project_path(get_value('science.tables.features_049', 'features_049.csv.gz'))))
AUDIT_DIR = Path(str(project_path(get_value('science.paths.directory_106', 'directory_106'))))
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MODEL_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
SAMPLE_COL = None
NORMALIZE_SAMPLE_ID_TO_LOWER = True
EPS = 1e-12

def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None

def read_table(path, sheet_name=0):
    path = Path(path)
    name_lower = path.name.lower()
    if name_lower.endswith(('.xlsx', '.xls')):
        return pd.read_excel(path, sheet_name=sheet_name)
    if name_lower.endswith(('.csv', '.csv.gz', '.gz')):
        return pd.read_csv(path, encoding='utf-8-sig')
    if name_lower.endswith(('.tsv', '.txt')):
        return pd.read_csv(path, sep='\t', encoding='utf-8-sig')
    raise ValueError(f'Unsupported file type: {path}')

def normalize_sample_id(value):
    if pd.isna(value):
        return ''
    text = str(value).strip().strip('"').strip("'")
    if re.fullmatch('-?\\d+\\.0', text):
        text = text[:-2]
    if NORMALIZE_SAMPLE_ID_TO_LOWER:
        text = text.lower()
    return text

def detect_sample_column(df):
    candidates = ['sample', 'sample_id', 'sample id', 'patient', 'patient_id', 'patient id', 'case', 'case_id', 'case id', 'subject', 'subject_id', 'subject id', 'id']
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]
    return df.columns[0]

def find_sample_column_in_model_ready(df):
    candidates = ['sample', 'sample_id', 'sample id', 'patient', 'patient_id', 'patient id', 'case', 'case_id', 'case id', 'subject', 'subject_id', 'subject id', 'id']
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]
    return df.columns[0]
if not EXTERNAL_INPUT_PATH.exists():
    raise FileNotFoundError(f'External radiomics file not found:\n{EXTERNAL_INPUT_PATH}')
PARAMETER_PATH = first_existing(PARAMETER_CANDIDATES)
if PARAMETER_PATH is None:
    raise FileNotFoundError(get_value('science.messages.status_024', 'input was not found.\nChecked:\n- ') + '\n- '.join((str(x) for x in PARAMETER_CANDIDATES)))
INTERNAL_MODEL_READY_PATH = first_existing(INTERNAL_MODEL_READY_CANDIDATES)
params = pd.read_csv(PARAMETER_PATH, encoding='utf-8-sig')
required_parameter_columns = ['feature', 'median_internal', 'mean_internal_after_imputation', 'scale_internal']
missing_parameter_columns = [col for col in required_parameter_columns if col not in params.columns]
if missing_parameter_columns:
    raise ValueError(f'Frozen parameter file is missing columns: {missing_parameter_columns}')
params = params.copy()
params['feature'] = params['feature'].astype(str)
if params['feature'].duplicated().any():
    duplicated = params.loc[params['feature'].duplicated(False), 'feature'].tolist()
    raise ValueError(f'Duplicated features in frozen parameter file: {duplicated[:20]}')
for col in ['median_internal', 'mean_internal_after_imputation', 'scale_internal']:
    params[col] = pd.to_numeric(params[col], errors='coerce')
if params[required_parameter_columns].isna().any().any():
    bad_rows = params.loc[params[required_parameter_columns].isna().any(axis=1), required_parameter_columns]
    raise ValueError(f'Frozen parameter file contains NA/non-numeric values. Examples:\n{bad_rows.head(10)}')
if (~np.isfinite(params['median_internal'])).any():
    raise ValueError('Non-finite internal medians were found.')
if (~np.isfinite(params['mean_internal_after_imputation'])).any():
    raise ValueError('Non-finite internal means were found.')
if (~np.isfinite(params['scale_internal']) | (params['scale_internal'] <= EPS)).any():
    bad = params.loc[~np.isfinite(params['scale_internal']) | (params['scale_internal'] <= EPS), ['feature', 'scale_internal']]
    raise ValueError(f'Invalid internal scales were found. Examples:\n{bad.head(20)}')
frozen_features = params['feature'].tolist()
median_internal = params.set_index('feature')['median_internal']
mean_internal = params.set_index('feature')['mean_internal_after_imputation']
scale_internal = params.set_index('feature')['scale_internal']

print('Frozen feature count:', len(frozen_features))
internal_feature_order_verified = False
if INTERNAL_MODEL_READY_PATH is not None:
    internal_ready = pd.read_csv(INTERNAL_MODEL_READY_PATH, nrows=2, encoding='utf-8-sig')
    internal_sample_col = find_sample_column_in_model_ready(internal_ready)
    internal_features = [str(c) for c in internal_ready.columns if c != internal_sample_col]
    if internal_features != frozen_features:
        missing_from_internal = [x for x in frozen_features if x not in internal_features]
        extra_in_internal = [x for x in internal_features if x not in frozen_features]
        raise ValueError(f"{get_value('science.messages.status_025', 'Feature order/content mismatch between the frozen parameter file and input.\nMissing from internal model-ready: ')}{missing_from_internal[:20]}\nExtra in internal model-ready: {extra_in_internal[:20]}")
    internal_feature_order_verified = True
    print('Internal feature-order check passed:', INTERNAL_MODEL_READY_PATH)
else:
    print(get_value('science.messages.status_026', 'Warning: input was not found; the parameter file will be used as the sole feature-order source.'))
raw = read_table(EXTERNAL_INPUT_PATH, sheet_name=SHEET_NAME)
if raw.empty:
    raise ValueError('The external radiomics table is empty.')
if raw.columns.duplicated().any():
    duplicated = raw.columns[raw.columns.duplicated(False)].astype(str).tolist()
    raise ValueError(f'Duplicated column names were detected in the external table: {duplicated[:20]}')
sample_col = SAMPLE_COL if SAMPLE_COL is not None else detect_sample_column(raw)
if sample_col not in raw.columns:
    raise ValueError(f'Specified sample column not found: {sample_col}')
sample_original = raw[sample_col].copy()
sample_normalized = sample_original.map(normalize_sample_id)
if (sample_normalized == '').any():
    bad_rows = np.where(sample_normalized.to_numpy() == '')[0].tolist()
    raise ValueError(f'Empty sample IDs remain after normalization. Row positions: {bad_rows[:20]}')
if sample_normalized.duplicated().any():
    duplicated_ids = sample_normalized[sample_normalized.duplicated(False)].tolist()
    raise ValueError(f'Duplicated sample IDs after normalization: {duplicated_ids[:20]}')
external_columns = [str(c) for c in raw.columns]
missing_required_features = [feature for feature in frozen_features if feature not in external_columns]
if missing_required_features:
    missing_path = AUDIT_DIR / get_value('science.tables.result_182', 'result_182.csv')
    pd.DataFrame({'missing_required_feature': missing_required_features}).to_csv(missing_path, index=False, encoding='utf-8-sig')
    raise ValueError(f'External file is missing {len(missing_required_features)} internally selected features. No output matrix was produced.\nExamples: {missing_required_features[:20]}\nFull list saved to: {missing_path}')
extra_external_columns = [col for col in external_columns if col != str(sample_col) and col not in frozen_features]
X_external_raw = raw.loc[:, frozen_features].copy()
X_external_numeric = X_external_raw.apply(pd.to_numeric, errors='coerce')
X_external_numeric = X_external_numeric.replace([np.inf, -np.inf], np.nan)
X_external_numeric.index = sample_normalized
X_external_numeric.index.name = 'sample'
missing_before = X_external_numeric.isna()
coercion_to_na = X_external_raw.notna().to_numpy() & X_external_numeric.reset_index(drop=True).isna().to_numpy()
n_coercion_to_na = int(coercion_to_na.sum())
print('External raw table shape:', raw.shape)
print('Detected sample column:', sample_col)
print('External samples:', len(X_external_numeric))
print('Required frozen features:', len(frozen_features))
print('Extra external columns ignored:', len(extra_external_columns))
print('Missing cells before internal-median imputation:', int(missing_before.sum().sum()))
print('Non-numeric cells converted to NA:', n_coercion_to_na)
X_external_imputed = X_external_numeric.fillna(median_internal)
if X_external_imputed.isna().any().any():
    raise ValueError('NA values remain after applying internal medians.')
X_external_scaled = X_external_imputed.sub(mean_internal, axis=1).div(scale_internal, axis=1)
X_external_scaled = X_external_scaled.loc[:, frozen_features]
if not np.isfinite(X_external_scaled.to_numpy()).all():
    bad_mask = ~np.isfinite(X_external_scaled.to_numpy())
    bad_rows, bad_cols = np.where(bad_mask)
    examples = [{'sample': X_external_scaled.index[r], 'feature': X_external_scaled.columns[c], 'value': X_external_scaled.iat[r, c]} for r, c in zip(bad_rows[:20], bad_cols[:20])]
    raise ValueError(f'NaN or infinite values remain after frozen preprocessing. Examples: {examples}')
model_ready = X_external_scaled.reset_index()
model_ready.to_csv(OUTPUT_MODEL_READY_PATH, index=False, encoding='utf-8-sig', compression='gzip')
check = pd.read_csv(OUTPUT_MODEL_READY_PATH, encoding='utf-8-sig')
expected_columns = ['sample'] + frozen_features
if check.columns.tolist() != expected_columns:
    raise ValueError('Saved external model-ready column order is incorrect.')
if check.shape != model_ready.shape:
    raise ValueError(f'Saved matrix shape changed unexpectedly: {check.shape} vs {model_ready.shape}')
if check['sample'].astype(str).tolist() != model_ready['sample'].astype(str).tolist():
    raise ValueError('Saved sample order changed unexpectedly.')
sample_qc = pd.DataFrame({'sample': X_external_numeric.index, 'original_sample_id': sample_original.astype(str).values, 'n_frozen_features': len(frozen_features), 'n_missing_before_imputation': missing_before.sum(axis=1).values, 'missing_fraction_before_imputation': missing_before.mean(axis=1).values, 'max_abs_z_after_frozen_scaling': X_external_scaled.abs().max(axis=1).values})
sample_qc_path = AUDIT_DIR / get_value('science.tables.result_183', 'result_183.csv')
sample_qc.to_csv(sample_qc_path, index=False, encoding='utf-8-sig')
extra_columns_path = AUDIT_DIR / get_value('science.tables.result_184', 'result_184.csv')
pd.DataFrame({'extra_external_column_ignored': extra_external_columns}).to_csv(extra_columns_path, index=False, encoding='utf-8-sig')
feature_qc = pd.DataFrame({'feature': frozen_features, 'external_missing_n_before_imputation': missing_before.sum(axis=0).reindex(frozen_features).values, 'external_missing_rate_before_imputation': missing_before.mean(axis=0).reindex(frozen_features).values, 'internal_median_applied': median_internal.reindex(frozen_features).values, 'internal_mean_applied': mean_internal.reindex(frozen_features).values, 'internal_scale_applied': scale_internal.reindex(frozen_features).values, 'external_mean_after_frozen_scaling': X_external_scaled.mean(axis=0).reindex(frozen_features).values, 'external_sd_after_frozen_scaling': X_external_scaled.std(axis=0, ddof=0).reindex(frozen_features).values})
feature_qc_path = AUDIT_DIR / get_value('science.tables.result_185', 'result_185.csv.gz')
feature_qc.to_csv(feature_qc_path, index=False, encoding='utf-8-sig', compression='gzip')
extreme_mask = X_external_scaled.abs() > 10
extreme_records = []
if extreme_mask.any().any():
    row_idx, col_idx = np.where(extreme_mask.to_numpy())
    for r, c in zip(row_idx, col_idx):
        extreme_records.append({'sample': X_external_scaled.index[r], 'feature': X_external_scaled.columns[c], 'z_value': X_external_scaled.iat[r, c]})
extreme_path = AUDIT_DIR / get_value('science.tables.result_186', 'result_186.csv')
pd.DataFrame(extreme_records, columns=['sample', 'feature', 'z_value']).to_csv(extreme_path, index=False, encoding='utf-8-sig')
audit = {'external_input_path': str(EXTERNAL_INPUT_PATH), 'sheet_name': SHEET_NAME, 'frozen_parameter_path': str(PARAMETER_PATH), 'internal_model_ready_path_used_for_order_check': str(INTERNAL_MODEL_READY_PATH) if INTERNAL_MODEL_READY_PATH is not None else None, 'internal_feature_order_verified': bool(internal_feature_order_verified), 'output_model_ready_path': str(OUTPUT_MODEL_READY_PATH), 'sample_column': str(sample_col), 'external_sample_n': int(X_external_scaled.shape[0]), 'frozen_feature_n': int(X_external_scaled.shape[1]), 'missing_required_feature_n': 0, 'extra_external_column_n_ignored': int(len(extra_external_columns)), 'missing_cell_n_before_internal_median_imputation': int(missing_before.sum().sum()), 'non_numeric_cell_n_converted_to_na': int(n_coercion_to_na), 'missing_cell_n_after_imputation': int(X_external_imputed.isna().sum().sum()), 'non_finite_cell_n_after_scaling': int((~np.isfinite(X_external_scaled.to_numpy())).sum()), 'abs_z_gt_10_cell_n': int(extreme_mask.sum().sum()), 'external_mean_max_abs_after_frozen_scaling': float(X_external_scaled.mean(axis=0).abs().max()), 'external_sd_min_after_frozen_scaling': float(X_external_scaled.std(axis=0, ddof=0).min()), 'external_sd_max_after_frozen_scaling': float(X_external_scaled.std(axis=0, ddof=0).max()), 'external_refitting_performed': False, 'outcome_labels_used': False, 'note': 'External feature means and SDs are not expected to equal 0 and 1, because internal cohort means and scales were applied without refitting.'}
audit_path = AUDIT_DIR / get_value('science.tables.result_187', 'result_187.json')
with open(audit_path, 'w', encoding='utf-8') as file:
    json.dump(audit, file, ensure_ascii=False, indent=2)



print('External model-ready matrix:', X_external_scaled.shape)






print('\nNo external preprocessing parameter was refitted.')
